# 06 | Burkina Faso: preparing WorldCereal EO extractions
**Ibrahim Kabore · WorldCereal extension · Koumbia, Burkina Faso**

Rather than generate thousands of satellite time series from scratch, I discovered existing public WorldCereal extractions for the five study years. This notebook downloads them, checks the satellite variables, decodes crop labels, aligns the temporal context and records validation groups. The resulting dataset is the input to Notebook 07.

**Sources:** [WorldCereal RDM](https://rdm.esa-worldcereal.org/) · [WorldCereal classification code](https://github.com/WorldCereal/worldcereal-classification) · [JECAM-CIRAD](https://doi.org/10.5194/essd-13-5951-2021). No openEO processing job is submitted. CPU runtime is sufficient.

## 1. Load project paths
This notebook is independent of the active runtime of Notebook 05. It uses the same Drive folder.

In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import requests

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/ESA_EO_Projects/WorldCereal_Burkina_Faso")
DATA = PROJECT / "data"
for folder in ["data", "notebooks", "models", "outputs"]:
    (PROJECT / folder).mkdir(parents=True, exist_ok=True)
print(PROJECT)

## 2. Discover public reference collections
Query the catalogue with the geographical extent of the Koumbia dataset rather than relying on the paginated list endpoint. A reference collection is not itself a satellite extraction.

In [ ]:
RDM = "https://ewoc-rdm-api.iiasa.ac.at/collections/search"
response = requests.get(RDM, params={"Bbox": [-3.90,10.96,-3.41,11.38]}, timeout=120)
response.raise_for_status()
collections = response.json()
results = pd.DataFrame([{"collection_id": c["collectionId"], "title": c["title"],
                         "samples": c["featureCount"], "access": c["accessType"]}
                        for c in collections])
display(results.sort_values("collection_id"))
display(results[results.collection_id.str.contains("bfa_jecam",case=False)])

## 3. Download existing monthly extractions
Use the verified public files discovered during the original run, retaining one file per reference year. Download only missing or invalid files. Validate the Parquet markers before saving.

In [ ]:
from urllib.parse import quote
BASE_URL = "https://s3.waw3-1.cloudferro.com/swift/v1/geoparquet/"
EXTRACTIONS = DATA / "extractions"
EXTRACTIONS.mkdir(exist_ok=True)
YEARS = [2017,2018,2020,2021,2022]
def parquet_ok(path):
    if not path.is_file() or path.stat().st_size < 8:
        return False
    with path.open("rb") as f:
        start = f.read(4)
        f.seek(-4, 2)
        return start == b"PAR1" and f.read(4) == b"PAR1"

paths = []
for year in YEARS:
    ref_id = f"{year}_BFA_JECAM-CIRAD_POLY_111"
    name = f"{ref_id}.parquet"
    path = EXTRACTIONS / name
    if not parquet_ok(path):
        relative = f"worldcereal_public_extractions.parquet/ref_id={ref_id}/{name}"
        response = requests.get(BASE_URL + quote(relative, safe="/="), timeout=180)
        response.raise_for_status()
        assert response.content[:4] == response.content[-4:] == b"PAR1"
        path.write_bytes(response.content)
    paths.append(path)
    print(name, f"{path.stat().st_size/1e6:.2f} MB")

## 4. Inspect the satellite data
Each sample has multiple monthly records. A row is a time step, not a distinct field. Verify that all 16 expected variables are available in every file.

In [ ]:
EXPECTED_BANDS = [f"S2-L2A-{b}" for b in ["B02","B03","B04","B05","B06","B07","B08","B8A","B11","B12"]] + [
    "S1-SIGMA0-VV","S1-SIGMA0-VH","AGERA5-TMEAN","AGERA5-PRECIP","elevation","slope"]
KEYS = ["ref_id","sample_id"]
tables = []
for path in paths:
    table = pd.read_parquet(path)
    assert set(EXPECTED_BANDS + KEYS + ["timestamp","valid_time","ewoc_code","lon","lat"]) <= set(table.columns)
    print(path.name, table.shape, "| unique samples:", table.sample_id.nunique())
    tables.append(table)
raw = pd.concat(tables, ignore_index=True)
print("Monthly rows:", len(raw))
print("Unique sample keys:", len(raw[KEYS].drop_duplicates()))
print("Columns:", raw.columns.tolist())
display(raw.head())

## 5. Decode crop categories
Use the official numerical EWOC legend. Preserve the six pure-crop labels and exclude mixtures or agroforestry classes instead of relabelling them as monocrops. Verify the exact codes against the downloaded legend.

In [ ]:
LEGEND_URL = "https://auxdata.terrascope.be/worldcereal/legend/WorldCereal_LC_CT_legend_latest.csv"
legend = pd.read_csv(LEGEND_URL, sep=";")
legend["ewoc_code"] = legend.ewoc_code.astype(str).str.replace("-","",regex=False).astype("int64")
assert legend.ewoc_code.is_unique
raw["ewoc_code"] = raw.ewoc_code.astype("int64")
assert raw.groupby(KEYS).ewoc_code.nunique().eq(1).all()
meta = raw.drop_duplicates(KEYS).copy()
counts = meta.groupby("ewoc_code").size().rename("samples").reset_index()
display(counts.merge(legend[["ewoc_code","label_full","level_1"]], on="ewoc_code", how="left").sort_values("samples",ascending=False))
CROP_MAP = {1101060000:"Maize",1108000010:"Cotton",1101070000:"Sorghum",
            1106000020:"Soybean",1105010050:"Cowpea",1106000050:"Groundnut"}
display(legend.loc[legend.ewoc_code.isin(CROP_MAP), ["ewoc_code","label_full","level_1"]])
assert set(CROP_MAP) <= set(legend.ewoc_code)
assert legend.loc[legend.ewoc_code.isin(CROP_MAP),"level_1"].eq("temporary_crops").all()

## 6. Align calendar months to the label year
The files may contain months before and after the survey year. Keep January through December of the label year; the initial seasonal pooling in Notebook 07 will be May–October. Check unique monthly slots, not merely row counts.

In [ ]:
raw["timestamp"] = pd.to_datetime(raw.timestamp, utc=True).dt.tz_localize(None)
raw["label_year"] = pd.to_datetime(raw.valid_time, utc=True).dt.year
assert raw.groupby(KEYS).label_year.nunique().eq(1).all()
annual = raw.loc[raw.timestamp.dt.year.eq(raw.label_year)].copy()
annual["month"] = annual.timestamp.dt.month
coverage = annual.groupby(KEYS).month.nunique()
display(coverage.value_counts().sort_index())
complete = coverage[coverage.eq(12)].reset_index()[KEYS]
annual = annual.merge(complete, on=KEYS, how="inner", validate="many_to_one")
annual["label"] = annual.ewoc_code.map(CROP_MAP)
crop_data = annual.loc[annual.label.notna()].copy()
assert not crop_data.duplicated(KEYS + ["month"]).any(), "Inspect duplicate monthly observations before continuing"
crop_meta = crop_data.drop_duplicates(KEYS)
print("Selected sample observations:", len(crop_meta))
display(pd.crosstab(crop_meta.label, crop_meta.label_year, margins=True))

## 7. Sensor availability and sample quality
Raw value 65535 indicates missing data. Check optical and radar availability separately, including by year. Mask missing measurements rather than inventing reflectances. A calendar slot can exist even when sensor measurements are missing.

In [ ]:
valid = crop_data[EXPECTED_BANDS].apply(pd.to_numeric, errors="coerce")
valid_mask = np.isfinite(valid) & valid.ne(65535)
for b in ["S1-SIGMA0-VV","S1-SIGMA0-VH"]:
    valid_mask[b] &= valid[b].gt(0)
quality = pd.DataFrame({"valid_pct":100*valid_mask.mean(), "missing_pct":100*(~valid_mask).mean()})
display(quality.round(2))
sensor_bands = ["S2-L2A-B04","S2-L2A-B08","S1-SIGMA0-VV","S1-SIGMA0-VH"]
display((100*valid_mask[sensor_bands].groupby(crop_data.label_year).mean()).round(2))
crop_data["observable"] = valid_mask["S2-L2A-B08"] | valid_mask["S1-SIGMA0-VV"]
annual_quality = crop_data.groupby(KEYS).observable.sum().rename("valid_months")
season_quality = crop_data.loc[crop_data.month.between(5,10)].groupby(KEYS).observable.sum().rename("valid_season_months")
qc = pd.concat([annual_quality,season_quality],axis=1).fillna(0)
usable = qc.loc[(qc.valid_months>=6) & (qc.valid_season_months>=3)].reset_index()
clean = crop_data.merge(usable[KEYS], on=KEYS, how="inner", validate="many_to_one")
print("Usable:", len(usable), "Excluded:", len(crop_meta)-len(usable))

## 8. Assign temporal splits and spatial blocks
These extracted points are not the original buffered JECAM points. Recompute block IDs from their own longitude and latitude. Keep all months of a sample in the same split. The temporal holdout measures year transfer, not independent spatial generalization.

In [ ]:
SPLIT = {2017: "train", 2018: "train", 2020: "train", 2021: "validation", 2022: "test"}
clean["split"] = clean.label_year.map(SPLIT)
assert clean.split.notna().all()
metadata = clean.drop_duplicates(KEYS).copy()
assert np.isfinite(metadata[["lon","lat"]]).all().all()
points = gpd.GeoDataFrame(metadata[KEYS+["label","label_year","split","lon","lat"]].copy(),
                         geometry=gpd.points_from_xy(metadata.lon,metadata.lat),
                         crs="EPSG:4326").to_crs("EPSG:32630")
points["spatial_group"] = ((points.geometry.x//2000).astype(int).astype(str)+"_"+
                           (points.geometry.y//2000).astype(int).astype(str))
clean = clean.merge(points[KEYS+["spatial_group"]], on=KEYS, validate="many_to_one")
assert clean.spatial_group.notna().all()
meta = clean.drop_duplicates(KEYS)
print("Spatial blocks:", meta.spatial_group.nunique())
display(pd.crosstab(meta.label,meta.split,margins=True))
ax = points.plot(column="label", cmap="tab10", legend=True, markersize=5, figsize=(12,8))
ax.set(title="WorldCereal extracted sample locations",xlabel="Easting (m)",ylabel="Northing (m)")
plt.show()

## 9. Save the dataset for Presto
Keep the exact input column names, sample identifiers, coordinates and labels needed by Notebook 07. Store both the monthly data and the quality/metadata tables. The next notebook will use the same frozen encoder as France but a different seasonal window.

In [ ]:
clean.to_parquet(DATA / "burkina_extractions_clean.parquet", index=False)
meta[KEYS+["label","label_year","split","lon","lat","spatial_group","ewoc_code"]].to_parquet(DATA / "burkina_extractions_metadata.parquet", index=False)
quality.to_csv(DATA / "burkina_extractions_band_quality.csv")
legend.to_csv(DATA / "worldcereal_crop_legend.csv", index=False)
print("Saved:", DATA / "burkina_extractions_clean.parquet")
print("Rows:", len(clean), "| samples:", len(meta))

## 10. Results from the original run and next experiment
The original run downloaded 52,072 monthly rows from 2,650 sample IDs. All had 12 calendar slots. The six pure crop classes contained 1,525 samples (18,300 monthly rows), all passing the quality threshold. The temporal split was 992 train, 271 validation and 262 test, with 32 spatial blocks. Optical availability was approximately 92.46%; radar availability was about 61.5% overall, but zero in the 2020 and 2021 extractions. This is uneven data coverage, not evidence that radar imagery did not exist in those years.

In Notebook 07, compare two configurations with identical sample splits: optical + climate + terrain (radar masked everywhere), and multisensor input (radar included wherever valid). Use May–October pooling, January–December context, masked B8A, and the verified August 2026 Presto encoder. Check preprocessing units and missing-value handling before inference. The buffered 2,964-point dataset from Notebook 05 remains separate. No accuracy result has been established for Burkina Faso yet.